# Generalización y protocolo experimental

**Capítulo 3 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_linear-regression/generalization.ipynb` · [Lección original](https://d2l.ai/chapter_linear-regression/generalization.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Generalización
<a id="sec_generalization_basics"></a>

Imaginemos dos estudiantes que preparan un examen con pruebas de años anteriores. Ellie memoriza todas las respuestas; Irene intenta descubrir las reglas que permiten resolverlas. Si el examen repite las preguntas, la memoria de Ellie puede darle una ventaja incluso frente a una regla que acierte el 90 % de los casos. Pero ante preguntas nuevas, recordar respuestas anteriores deja de ser suficiente. Irene puede seguir aplicando los patrones aprendidos. La distinción no está entre recordar mucho o poco, sino entre reproducir casos vistos y resolver casos nuevos.

Como científicos de aprendizaje automático, nuestro objetivo es descubrir *patrones*. ¿Pero cómo podemos estar seguros de que hemos descubierto realmente un patrón *general* y no simplemente memorizar nuestros datos? La mayoría de las veces, nuestras predicciones sólo son útiles si nuestro modelo descubre un patrón. No queremos predecir los precios de las acciones de ayer, sino de mañana. No necesitamos reconocer enfermedades ya diagnosticadas para pacientes previamente vistos, sino más bien enfermedades previamente no diagnosticadas en pacientes no vistos anteriormente. Este problema -cómo descubrir patrones que *generalizan*- es el problema fundamental del aprendizaje automático, y posiblemente de todas las estadísticas. Podríamos plantear este problema como una sola parte de una pregunta mucho más grande que envuelve toda la ciencia: ¿cuándo estamos justificados para dar el salto de observaciones particulares a declaraciones más generales?

En la vida real, debemos adaptarnos a nuestros modelos utilizando una colección finita de datos. Las escalas típicas de esos datos varían enormemente entre los dominios. Para muchos problemas médicos importantes, sólo podemos acceder a unos pocos miles de puntos de datos. Al estudiar enfermedades raras, podríamos tener suerte de acceder a cientos. Por el contrario, los conjuntos de datos públicos más grandes consistentes en fotografías etiquetadas, por ejemplo, ImageNet [Deng.Dong.Socher.ea.2009](https://d2l.ai/chapter_references/zreferences.html), contienen millones de imágenes. Y algunas colecciones de imágenes sin etiqueta, como el conjunto de datos Flickr YFC100M, pueden ser aún más grandes, conteniendo más de 100 millones de imágenes [thomee2016yfcc100m](https://d2l.ai/chapter_references/zreferences.html). Sin embargo, incluso a esta escala extrema, el número de puntos de datos disponibles sigue siendo infinitamente pequeño en comparación con el espacio de todas las imágenes posibles en una resolución de megapíxeles.

El fenómeno de la adaptación más cercana a nuestros datos de entrenamiento que a la distribución subyacente se llama *overfitting*, y las técnicas para combatir el sobrefitting se llaman a menudo métodos *regularización*. Si bien no es sustituto de una introducción adecuada a la teoría del aprendizaje estadístico (ver [Vapnik98,boucheron2005theory](https://d2l.ai/chapter_references/zreferences.html)), le daremos la intuición suficiente para ponerse en marcha. Revisaremos la generalización en muchos capítulos a través del libro, explorando tanto lo que se sabe acerca de los principios subyacentes a la generalización en varios modelos, y también las técnicas heurísticas que se han encontrado (empíricamente) para dar lugar a una mejor generalización en las tareas de interés práctico.

## Error de entrenamiento y generalización
En la configuración estándar de aprendizaje supervisado, suponemos que los datos de entrenamiento y los datos de prueba se extraen *de forma independiente* de distribuciones *idénticas*. Esto se llama comúnmente la suposición *ID*. Si bien esta suposición es fuerte, vale la pena señalar que, a falta de tal suposición, estaríamos muertos en el agua. ¿Por qué deberíamos creer que los datos de entrenamiento muestreados de distribución $P(X,Y)$ deberían decirnos cómo hacer predicciones sobre datos de prueba generados por una distribución *diferente* $Q(X,Y)$? Hacer tales saltos resulta requerir fuertes suposiciones sobre cómo se relacionan $P$ y $Q$. Más adelante discutiremos algunas suposiciones que permiten cambios en la distribución pero primero necesitamos entender el caso de IID, donde $P(\cdot) = Q(\cdot)$.

Para empezar, necesitamos diferenciar entre el *error de entrenamiento* $R_\textrm{emp}$, que es un *estadístico* calculado en el conjunto de datos de entrenamiento, y el *error de generalización* $R$, que es una *expectación* tomada con respecto a la distribución subyacente. Puede pensar en el error de generalización como lo que vería si aplicara su modelo a un flujo infinito de ejemplos de datos adicionales extraídos de la misma distribución de datos subyacente. Formalmente el error de entrenamiento se expresa como una *suma* (con la misma notación que [Referencia sec_linear_regression](https://d2l.ai/chapter_linear-regression/linear-regression.html#sec-linear-regression)):

$$R_\textrm{emp}[\mathbf{X}, \mathbf{y}, f] = \frac{1}{n} \sum_{i=1}^n l(\mathbf{x}^{(i)}, y^{(i)}, f(\mathbf{x}^{(i)})),$$

mientras que el error de generalización se expresa como integral:

$$R[p, f] = E_{(\mathbf{x}, y) \sim P} [l(\mathbf{x}, y, f(\mathbf{x}))] =
\int \int l(\mathbf{x}, y, f(\mathbf{x})) p(\mathbf{x}, y) \;d\mathbf{x} dy.$$

Problemáticamente, nunca podemos calcular el error de generalización $R$ exactamente. Nadie nos dice nunca la forma precisa de la función de densidad $p(\mathbf{x}, y)$. Además, no podemos probar un flujo infinito de puntos de datos. Así, en la práctica, debemos *estimar* el error de generalización mediante la aplicación de nuestro modelo a un conjunto de pruebas independiente constituido por una selección aleatoria de ejemplos $\mathbf{X}'$ y etiquetas $\mathbf{y}'$ que se retuvieron de nuestro conjunto de entrenamiento. Esto consiste en aplicar la misma fórmula que se utilizó para calcular el error de entrenamiento empírico pero a un conjunto de pruebas $\mathbf{X}', \mathbf{y}'$.

Es crucial, cuando evaluamos nuestro clasificador en el conjunto de pruebas, estamos trabajando con un clasificador *fijo* (no depende de la muestra del conjunto de pruebas), y por lo tanto estimar su error es simplemente el problema de la estimación media. Sin embargo, no se puede decir lo mismo para el conjunto de entrenamiento. Tenga en cuenta que el modelo con el que terminamos depende explícitamente de la selección del conjunto de entrenamiento y por lo tanto el error de entrenamiento será en general una estimación sesgada del verdadero error en la población subyacente. La cuestión central de la generalización es entonces cuando debemos esperar que nuestro error de entrenamiento esté cerca del error de población (y por lo tanto el error de generalización).

### Complejidad del modelo
En teoría clásica, cuando tenemos modelos simples y datos abundantes, los errores de entrenamiento y generalización tienden a estar cerca. Sin embargo, cuando trabajamos con modelos más complejos y/o con menos ejemplos, esperamos que el error de entrenamiento baje pero que crezca la brecha de generalización. Esto no debería ser sorprendente. Imaginemos una clase de modelo tan expresiva que para cualquier conjunto de datos de ejemplos $n$, podamos encontrar un conjunto de parámetros que puedan encajar perfectamente con etiquetas arbitrarias, incluso si se asignan aleatoriamente. En este caso, incluso si encajamos perfectamente nuestros datos de entrenamiento, ¿cómo podemos concluir algo sobre el error de generalización? Por lo que sabemos, nuestro error de generalización podría no ser mejor que adivinar aleatoriamente.

En general, en ausencia de cualquier restricción en nuestra clase modelo, no podemos concluir, basándose en el ajuste de los datos de entrenamiento solamente, que nuestro modelo ha descubierto cualquier patrón generalizable [vapnik1994measuring](https://d2l.ai/chapter_references/zreferences.html). Por otro lado, si nuestra clase modelo no era capaz de ajustar etiquetas arbitrarias, entonces debe haber descubierto un patrón. Ideas teóricas de aprendizaje sobre la complejidad del modelo derivaron alguna inspiración de las ideas de Karl Popper, un filósofo influyente de la ciencia, que formalizó el criterio de la falsedad. De acuerdo con Popper, una teoría que puede explicar cualquier observación no es una teoría científica en absoluto! Después de todo, ¿qué nos ha dicho sobre el mundo si no ha descartado ninguna posibilidad? En resumen, lo que queremos es una hipótesis que *no podría* explicar cualquier observación que podríamos hacer y sin embargo resulta compatible con esas observaciones que *de hecho* hacemos.

Ahora bien, lo que constituye precisamente una noción apropiada de complejidad del modelo es un asunto complejo. A menudo, los modelos con más parámetros son capaces de adaptarse a un mayor número de etiquetas asignadas arbitrariamente. Sin embargo, esto no es necesariamente cierto. Por ejemplo, los métodos de núcleo operan en espacios con un número infinito de parámetros, pero su complejidad se controla por otros medios [Scholkopf.Smola.2002](https://d2l.ai/chapter_references/zreferences.html). Una noción de complejidad que a menudo resulta útil es el rango de valores que pueden tomar los parámetros. Aquí, un modelo cuyos parámetros están permitidos tomar valores arbitrarios sería más complejo. Volveremos a esta idea en la siguiente sección, cuando introducimos *caída del peso*, su primera técnica práctica de regularización. Cabe destacar que puede ser difícil comparar la complejidad entre miembros de clases de modelos sustancialmente diferentes (por ejemplo, árboles de decisión vs. redes neuronales).

En este punto, debemos enfatizar otro punto importante que revisaremos cuando introducimos redes neuronales profundas. Cuando un modelo es capaz de ajustar etiquetas arbitrarias, el error de bajo entrenamiento no implica necesariamente un error de baja generalización. *Sin embargo, tampoco implica necesariamente un error de alta generalización!* Todo lo que podemos decir con confianza es que el error de bajo entrenamiento por sí solo no es suficiente para certificar un error de baja generalización. Las redes neuronales profundas resultan ser simplemente tales modelos: mientras que se generalizan bien en la práctica, son demasiado poderosas para permitirnos concluir mucho sobre la base del error de entrenamiento solo. En estos casos debemos confiar más en nuestros datos de retención para certificar la generalización después del hecho. Error en los datos de retención, es decir, conjunto de validación, se llama el *error de validación*.

## ¿Underfitting o Overfitting?
Cuando comparamos los errores de entrenamiento y validación, queremos ser conscientes de dos situaciones comunes. Primero, queremos tener cuidado con los casos en que nuestro error de entrenamiento y error de validación son tanto sustanciales pero hay una pequeña brecha entre ellos. Si el modelo es incapaz de reducir el error de entrenamiento, eso podría significar que nuestro modelo es demasiado simple (es decir, insuficientemente expresivo) para capturar el patrón que estamos tratando de modelar. Además, ya que la brecha de generalización * ($R_\textrm{emp} - R$) entre nuestros errores de entrenamiento y generalización es pequeña, tenemos razones para creer que podríamos salirnos con un modelo más complejo.

Por otro lado, como hemos comentado anteriormente, queremos tener cuidado con los casos en los que nuestro error de entrenamiento es significativamente menor que nuestro error de validación, lo que indica que el exceso de ajuste *. Tenga en cuenta que el exceso de ajuste no siempre es algo malo. En el aprendizaje profundo especialmente, los mejores modelos predictivos a menudo funcionan mucho mejor en los datos de entrenamiento que en los datos de retención. En última instancia, normalmente nos preocupamos por conducir el error de generalización más bajo, y sólo nos preocupamos por la brecha en la medida en que se convierte en un obstáculo para ese fin. Tenga en cuenta que si el error de entrenamiento es cero, entonces la brecha de generalización es exactamente igual al error de generalización y podemos progresar sólo reduciendo la brecha.

### Ajuste de curva polinómica
<a id="subsec_polynomial-curve-fitting"></a>

Para ilustrar una intuición clásica sobre el sobreajuste y la complejidad del modelo, considere lo siguiente: dados los datos de entrenamiento que consisten en una sola característica $x$ y una correspondiente etiqueta de valor real $y$, tratamos de encontrar el polinomio de grado $d$

$$\hat{y}= \sum_{i=0}^d x^i w_i$$

para estimar la etiqueta $y$. Esto es sólo un problema de regresión lineal donde nuestras características son dadas por los poderes de $x$, los pesos del modelo son dadas por $w_i$, y el sesgo es dado por $w_0$ desde $x^0 = 1$ para todo $x$. Puesto que esto es sólo un problema de regresión lineal, podemos utilizar el error cuadrado como nuestra función de pérdida.

Una función polinómica de orden superior es más compleja que una función polinómica de orden inferior, ya que el polinomio de orden superior tiene más parámetros y el rango de selección de la función de modelo es más amplio. Fijando el conjunto de datos de entrenamiento, las funciones polinómicas de orden superior siempre deben lograr un error de entrenamiento menor (en el peor de los casos, igual) en relación con los polinomios de grado inferior. De hecho, siempre que cada ejemplo de datos tenga un valor distinto de $x$, una función polinómica con un grado igual al número de ejemplos de datos puede encajar perfectamente con el conjunto de entrenamiento. Comparamos la relación entre el grado polinomio (complejidad del modelo) y tanto un ajuste inferior como un sobreajuste en [Referencia fig_capacity_vs_error](https://d2l.ai/chapter_linear-regression/generalization.html#fig-capacity-vs-error).

![Influencia de la complejidad del modelo sobre el infraajuste y el sobreajuste.](../recursos/originales/capacity-vs-error.svg)
<a id="fig_capacity_vs_error"></a>

### Tamaño del conjunto de datos
Como el límite anterior ya indica, otra gran consideración a tener en cuenta es el tamaño del conjunto de datos. Fijando nuestro modelo, cuanto menos muestras tengamos en el conjunto de datos de entrenamiento, más probable (y más severamente) que nos encontremos con exceso de ajuste. A medida que aumentamos la cantidad de datos de entrenamiento, el error de generalización típicamente disminuye. Además, en general, más datos nunca duelen. Para una tarea fija y distribución de datos, la complejidad del modelo no debe aumentar más rápidamente que la cantidad de datos. Dados más datos, podríamos intentar encajar en un modelo más complejo. Ausente datos suficientes, los modelos más simples pueden ser más difíciles de superar. Para muchas tareas, el aprendizaje profundo solo supera los modelos lineales cuando se dispone de muchos miles de ejemplos de entrenamiento. En parte, el éxito actual del aprendizaje profundo se debe considerablemente a la abundancia de conjuntos de datos masivos que surgen de empresas de Internet, almacenamiento barato, dispositivos conectados, y la amplia digitalización de la economía.

## Selección de modelos
<a id="subsec_generalization-model-selection"></a>

Por lo general, seleccionamos nuestro modelo final sólo después de evaluar múltiples modelos que difieren de varias maneras (arquitecturas diferentes, objetivos de formación, características seleccionadas, preprocesamiento de datos, tasas de aprendizaje, etc.).

En principio, no debemos tocar nuestro conjunto de pruebas hasta después de haber elegido todos nuestros hiperparametros. Si usamos los datos de prueba en el proceso de selección del modelo, existe el riesgo de que podamos sobreadaptarnos a los datos de prueba. Entonces estaríamos en serios problemas. Si sobreadaptamos nuestros datos de entrenamiento, siempre hay la evaluación de los datos de prueba para mantenernos honestos. Pero si sobreadaptamos los datos de prueba, ¿cómo lo sabríamos? Vea [ong2005learning](https://d2l.ai/chapter_references/zreferences.html) para un ejemplo de cómo esto puede conducir a resultados absurdos incluso para modelos donde la complejidad puede ser estrechamente controlada.

Por lo tanto, nunca debemos confiar en los datos de prueba para la selección de modelos. Y sin embargo, no podemos confiar únicamente en los datos de entrenamiento para la selección de modelos, ya sea porque no podemos estimar el error de generalización en los mismos datos que utilizamos para entrenar el modelo.

En las aplicaciones prácticas, la imagen se vuelve más confusa. Si bien lo ideal es que solo toquemos los datos de prueba una vez, para evaluar el mejor modelo o comparar un pequeño número de modelos entre sí, los datos de prueba del mundo real rara vez se descartan después de un solo uso. Rara vez podemos permitirnos un nuevo conjunto de pruebas para cada ronda de experimentos. De hecho, los datos de referencia de reciclaje durante décadas pueden tener un impacto significativo en el desarrollo de algoritmos, por ejemplo, para [image classification](https://paperswithcode.com/sota/image-classification-on-imagenet) y [optical character recognition](https://paperswithcode.com/sota/image-classification-on-mnist).

La práctica común para abordar el problema de *entrenamiento en el conjunto de pruebas* es dividir nuestros datos de tres maneras, incorporando un *conjunto de validación* además de los conjuntos de datos de entrenamiento y pruebas. El resultado es un negocio turbio donde los límites entre la validación y los datos de prueba son preocupantemente ambiguos. A menos que se indique explícitamente lo contrario, en los experimentos de este libro estamos trabajando realmente con lo que con razón debe llamarse datos de entrenamiento y datos de validación, sin conjuntos de pruebas verdaderos. Por lo tanto, la exactitud reportada en cada experimento del libro es realmente la exactitud de validación y no una verdadera precisión de conjunto de pruebas.

### Validación cruzada
Cuando los datos de entrenamiento son escasos, es posible que ni siquiera podamos darnos el lujo de mantener suficientes datos para constituir un conjunto de validación adecuado. Una solución popular a este problema es emplear $K$*-fold-cross-validation*. Aquí, los datos de entrenamiento originales se dividen en subconjuntos $K$ no superpuestos. A continuación, el entrenamiento y validación de modelos se ejecutan $K$ veces, cada vez que el entrenamiento en subconjuntos $K-1$ y la validación en un subconjunto diferente (el que no se utiliza para el entrenamiento en esa ronda). Finalmente, los errores de entrenamiento y validación se calculan promediando sobre los resultados de los experimentos $K$.

## Resumen
Esta sección explora algunos de los fundamentos de la generalización en el aprendizaje automático. Algunas de estas ideas se vuelven complicadas y contraintuitivas cuando llegamos a modelos más profundos; aquí, los modelos son capaces de sobreajustar los datos mal, y las nociones relevantes de complejidad pueden ser tanto implícitas como contraintuitivas (por ejemplo, arquitecturas más grandes con más parámetros generalizando mejor).

1. Utilizar conjuntos de validación (o $K$* de validación cruzada*) para la selección de modelos;
1. Los modelos más complejos a menudo requieren más datos;
1. Las nociones pertinentes de complejidad incluyen tanto el número de parámetros como la gama de valores que se les permite tomar;
1. Mantener todo lo demás igual, más datos casi siempre conduce a una mejor generalización;
1. Toda esta charla de generalización se basa en la suposición del IID. Si relajamos esta suposición, permitiendo que las distribuciones cambien entre el tren y los períodos de prueba, entonces no podemos decir nada acerca de la generalización ausente una suposición más (tal vez más leve).

## Ejercicios
1. ¿Cuándo se puede resolver exactamente el problema de la regresión polinómica?
1. Dé al menos cinco ejemplos donde las variables aleatorias dependientes hacen que el tratamiento del problema como datos IID no sea aconsejable.
1. ¿Alguna vez puede esperar ver cero error de entrenamiento? ¿En qué circunstancias vería cero error de generalización?
1. ¿Por qué es muy caro calcular la validación cruzada $K$-fold?
1. ¿Por qué la estimación de error de validación cruzada $K$-fold está sesgada?
1. La dimensión VC se define como el número máximo de puntos que se pueden clasificar con etiquetas arbitrarias $\{\pm 1\}$ por una función de una clase de funciones. ¿Por qué no podría esto ser una buena idea para medir lo compleja que es la clase de funciones? Consejo: considere la magnitud de las funciones.
1. Su gestor le da un conjunto de datos difícil en el que su algoritmo actual no funciona tan bien. ¿Cómo se justificaría a él que usted necesita más datos? Sugerencia: no se puede aumentar los datos, pero se puede disminuir.


### Nota docente de Hespérides

El error de entrenamiento mide ajuste a datos ya observados. El de validación orienta decisiones como complejidad y regularización; tras muchas decisiones también puede sobreajustarse esa validación. Early stopping guarda el estado con mejor criterio de validación y lo restaura, no entrega automáticamente la última época. El test se usa al final para estimar el comportamiento de la selección ya fijada. En series temporales, respeta además el orden de la información disponible.

Vínculo con los apuntes: sesión 3, «Generalización y protocolo experimental».


[Debate del original](https://discuss.d2l.ai/t/97)
